In [ ]:
from project_config import CATALOG_ROOT, SAMPLE_ROOT, RADAR_DOC_ROOT, NWP_DOC_ROOT, OUTPUT_ROOT, DATASET_SPLIT, load_pickle, write_sample_index
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
import numpy as np
import os
import pandas as pd
import datetime
from scipy.ndimage import gaussian_filter
from scipy.ndimage import distance_transform_edt
from scipy.interpolate import RegularGridInterpolator


In [ ]:
sta_dic = {}
data_root = RADAR_DOC_ROOT
file_list = os.listdir(data_root)
for i in range(len(file_list)):
    textid = file_list[i].split('.')[0].split('_')[2]
    sta_file = os.path.join(data_root, file_list[i])
    labelfmt = pd.read_csv(sta_file)
    lonss = labelfmt.iloc[0, 0].split('=')
    lons = round(float(lonss[1]), 4)
    lonee = labelfmt.iloc[2, 0].split('=')
    lone = round(float(lonee[1]), 4)
    latss = labelfmt.iloc[1, 0].split('=')
    lats = round(float(latss[1]), 4)
    latee = labelfmt.iloc[3, 0].split('=')
    late = round(float(latee[1]), 4)
    sta_dic[textid] = [lons, lone, lats, late]
sta_dic_nwp = {}
data_root = NWP_DOC_ROOT
file_list = os.listdir(data_root)
for i in range(len(file_list)):
    textid = file_list[i].split('.')[0].split('_')[2]
    sta_file = os.path.join(data_root, file_list[i])
    labelfmt = pd.read_csv(sta_file)
    lonss = labelfmt.iloc[0, 0].split('=')
    lons = round(float(lonss[1]), 4)
    lonee = labelfmt.iloc[2, 0].split('=')
    lone = round(float(lonee[1]), 4)
    latss = labelfmt.iloc[1, 0].split('=')
    lats = round(float(latss[1]), 4)
    latee = labelfmt.iloc[3, 0].split('=')
    late = round(float(latee[1]), 4)
    sta_dic_nwp[textid] = [lons, lone, lats, late]

def get_coord(stanum, coord, sta_dic, test=0):
    resol = 0.01
    [lons, lone, lats, late] = sta_dic[stanum]
    pix_lon = (coord[0] - lons) // resol
    pix_lat = 400 - (coord[1] - late) // resol
    pix_c = [int(pix_lon), int(pix_lat)]
    if test == 1:
        print('Grid coordinates', pix_c)
        print('Extent', lons, lone, lats, late)
        x1 = np.linspace(lons, lone, 401)
        y1 = np.linspace(late, lats, 401)[::-1]
        x1 = np.round(x1, 4)
        y1 = np.round(y1, 4)
        print('Input coordinates', coord)
        print('Matched coordinates', x1[pix_c[0]], y1[pix_c[1]])
        return (pix_c, [lons, lone, lats, late])
    elif 0 <= pix_c[0] < 401 and 0 <= pix_c[1] < 401:
        return pix_c
    else:
        return
data_dic = load_pickle(CATALOG_ROOT / DATASET_SPLIT / 'data_dic_nwp.pkl')
mask_dic_all = load_pickle(CATALOG_ROOT / DATASET_SPLIT / 'mask_dic_all.pkl')

def fill_nan_nearest(arr):
    nan_mask = np.isnan(arr)
    if not np.any(nan_mask):
        return arr.copy()
    indices = distance_transform_edt(nan_mask, return_distances=False, return_indices=True)
    filled = arr[tuple(indices)]
    return filled

def resize_linear(arr, new_shape):
    h_old, w_old = arr.shape
    h_new, w_new = new_shape
    y_old = np.arange(h_old)
    x_old = np.arange(w_old)
    y_new = np.linspace(0, h_old - 1, h_new)
    x_new = np.linspace(0, w_old - 1, w_new)
    Y_new, X_new = np.meshgrid(y_new, x_new, indexing='ij')
    interpolator = RegularGridInterpolator((y_old, x_old), arr, method='linear', bounds_error=False, fill_value=None)
    resized = interpolator(np.stack([Y_new, X_new], axis=-1))
    return resized


In [ ]:
input_len = 10
output_len = 20
save_dir = SAMPLE_ROOT / DATASET_SPLIT
radar_key_list = ['CR', 'VIL', 'EB20', 'ET20', 'EB30', 'ET30', 'EB45', 'ET45']
nwp_keylist = ['HT0', 'HT10', 'HT20', 'HTw0']
max_dic = {'CR': 100, 'VIL': 80, 'EB20': 32, 'ET20': 32, 'EB30': 32, 'ET30': 32, 'EB45': 32, 'ET45': 32}
max_dic_nwp = {'HT0': {'max': 6516.0, 'min': 172.1}, 'HT10': {'max': 8270.0, 'min': 812.0}, 'HT20': {'max': 9790.0, 'min': 2644.0}, 'HTw0': {'max': 6564.0, 'min': 133.0}}
broken_count = 0
train_dic = {}
iii = 0
for case_id in data_dic:
    data_len = len(data_dic[case_id]['CR'])
    if data_len < input_len + output_len:
        continue
    region_id = case_id.split('_')[1]
    stid = case_id.split('_')[3]
    start_i = 0
    broken = 0
    label_temp = mask_dic_all[case_id]
    st_loc = np.zeros((401, 401))
    for coord_text in label_temp:
        coord = label_temp[coord_text]
        st_loc[coord[1], coord[0]] = 1
    st_loc = gaussian_filter(st_loc, sigma=8)
    st_loc = np.round(st_loc * 400, 2)
    st_loc[st_loc > 1] = 1
    while start_i + input_len + output_len <= data_len:
        broken = 0
        input_file = []
        for key in radar_key_list:
            if key == 'label':
                continue
            for i in range(input_len):
                try:
                    data_temp = np.load(data_dic[case_id][key][start_i + i])
                except:
                    broken = 1
                    print(case_id)
                    break
                data_temp[data_temp < 0] = 0
                data_temp[data_temp > max_dic[key]] = max_dic[key]
                data_temp = np.round(data_temp / max_dic[key], 8)
                input_file.append(data_temp)
        nwp_starttime = data_dic[case_id]['CR'][start_i + input_len - 1].replace('\\', '/').split('/')[-1].split('.')[0].split('_')[3]
        nwp_starttime = datetime.datetime.strptime('2024' + nwp_starttime, '%Y%m%d-%H%M')
        index_nwp = None
        for candidate_index in range(len(data_dic[case_id]['HT0'])):
            timetemp_nwp = data_dic[case_id]['HT0'][candidate_index].replace('\\', '/').split('/')[-1].split('.')[0].split('_')[3]
            timetemp_nwp = datetime.datetime.strptime('2024' + timetemp_nwp, '%Y%m%d-%H%M')
            if timetemp_nwp.replace(minute=0) == nwp_starttime.replace(minute=0):
                index_nwp = candidate_index
                break
        if index_nwp is None:
            start_i += 1
            continue
        for key in nwp_keylist:
            for n in range(3):
                try:
                    data = np.load(data_dic[case_id][key][index_nwp + n])
                except:
                    broken = 1
                    print('BROKEN', case_id, start_i, nwp_starttime)
                    broken_count += 1
                    break
                data = fill_nan_nearest(data)
                data = np.asarray(data, dtype=np.float64)
                data = resize_linear(data, (501, 501))
                start_lat = int(np.abs(sta_dic_nwp[stid][2] - sta_dic[stid][2]) // 0.01)
                start_lon = int(np.abs(sta_dic_nwp[stid][0] - sta_dic[stid][0]) // 0.01)
                data = data[start_lat:start_lat + 401, start_lon:start_lon + 401]
                data = (data - max_dic_nwp[key]['min']) / (max_dic_nwp[key]['max'] - max_dic_nwp[key]['min'])
                input_file.append(data)
        for i in range(input_len):
            input_temp = np.zeros((401, 401))
            output_coord = data_dic[case_id]['label'][start_i + i]
            for j in range(len(output_coord)):
                input_temp[output_coord[j][1], output_coord[j][0]] = 1
            if len(output_coord) > 0:
                input_temp = gaussian_filter(input_temp, sigma=8)
                input_temp = np.round(input_temp * 400, 2)
                input_temp[input_temp > 1] = 1
            input_file.append(input_temp)
        if broken or len(input_file) != 102 or any((field.shape != (401, 401) for field in input_file)):
            start_i += 1
            continue
        input_file = np.array(input_file, dtype=np.float32)
        output_file = []
        for i in range(output_len):
            output_temp = np.zeros((401, 401))
            output_coord = data_dic[case_id]['label'][start_i + input_len + i]
            for j in range(len(output_coord)):
                output_temp[output_coord[j][1], output_coord[j][0]] = 1
            if len(output_coord) > 0:
                output_temp = gaussian_filter(output_temp, sigma=8)
                output_temp = np.round(output_temp * 400, 2)
                output_temp[output_temp > 1] = 1
            output_file.append(output_temp)
        output_file = np.array(output_file, dtype=np.float32)
        if broken == 0 and input_file.shape == (102, 401, 401):
            dir_temp = os.path.join(save_dir, region_id, case_id, 'input')
            if not os.path.exists(dir_temp):
                os.makedirs(dir_temp)
            save_name = os.path.join(dir_temp, str(start_i))
            np.savez_compressed(save_name, input_file)
            dir_temp = os.path.join(save_dir, region_id, case_id, 'mask')
            if not os.path.exists(dir_temp):
                os.makedirs(dir_temp)
            save_name = os.path.join(dir_temp, str(start_i))
            np.savez_compressed(save_name, st_loc)
            dir_temp = os.path.join(save_dir, region_id, case_id, 'output')
            if not os.path.exists(dir_temp):
                os.makedirs(dir_temp)
            save_name = os.path.join(dir_temp, str(start_i))
            np.savez_compressed(save_name, output_file)
        start_i += 1
    print(case_id)


In [ ]:
train_dic_matrix = write_sample_index(DATASET_SPLIT)
